In [1]:
# Recharge tout depuis data/ (aucun téléchargement)
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("data/dataset.csv", index_col=0, parse_dates=True)
stocks = pd.read_csv("data/eia_stocks.csv", index_col=0, parse_dates=True).squeeze("columns")
intra = pd.read_csv("data/eia_intraday.csv", index_col=0, parse_dates=True)

lvl = stocks.shift(1).set_axis(stocks.index + pd.Timedelta(days=5))
e = pd.DataFrame({
    "r": 100 * df["r"],
    "S": 100 * df["surprise"] / lvl.reindex(df.index),
    "B": (df["slope"].shift(1) > 0).astype(int),
}).dropna()
e2 = e.join(intra, how="inner")
print(len(e), "événements journaliers,", len(e2), "intraday")

736 événements journaliers, 491 intraday


In [2]:
# Robustesse de S:B : tendance temporelle, surprise en Mb, hors échantillon
e2["t"] = e2.index.year - 2013                          # années depuis 2013
e2["S_mb"] = df["surprise"].reindex(e2.index)           # surprise en millions de barils
out = {}
for k, f in {"tendance": "r5 ~ S * B + S:t + t", "en Mb": "r5 ~ S_mb * B"}.items():
    m = smf.ols(f, data=e2).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
    out[k] = pd.concat([m.params, m.pvalues], keys=["coef", "p"])
print(pd.DataFrame(out).round(3), "\n")

train, test = e2[:"2018"], e2["2019":]
for f in ["r5 ~ S", "r5 ~ S * B"]:
    pred = smf.ols(f, data=train).fit().predict(test)
    r2 = 1 - ((test["r5"] - pred) ** 2).sum() / ((test["r5"] - train["r5"].mean()) ** 2).sum()
    print(f"{f:12s} R2 hors échantillon = {r2:.4f}")

                tendance  en Mb
coef B            -0.026 -0.000
     Intercept    -0.086 -0.034
     S            -0.330    NaN
     S:B           0.096    NaN
     S:t           0.020    NaN
     S_mb            NaN -0.050
     S_mb:B          NaN  0.027
     t             0.013    NaN
p    B             0.610  0.994
     Intercept     0.095  0.382
     S             0.000    NaN
     S:B           0.017    NaN
     S:t           0.004    NaN
     S_mb            NaN  0.000
     S_mb:B          NaN  0.009
     t             0.053    NaN 

r5 ~ S       R2 hors échantillon = -0.0621
r5 ~ S * B   R2 hors échantillon = -0.0308


In [3]:
# HMM (spec du collègue) : probabilités FILTRÉES, paramètres estimés sur 2007-2018 uniquement
from hmmlearn.hmm import GaussianHMM
from scipy.stats import multivariate_normal
from sklearn.preprocessing import StandardScaler

feat = pd.read_csv("data/wti_hmm_features.csv", index_col=0, parse_dates=True).sort_index()
sc = StandardScaler().fit(feat[:"2018"])
X = sc.transform(feat)
hmm = GaussianHMM(3, covariance_type="full", n_iter=500, random_state=42).fit(X[feat.index <= "2018-12-31"])

ll = np.column_stack([multivariate_normal(hmm.means_[k], hmm.covars_[k]).logpdf(X) for k in range(3)])
a, p = np.zeros_like(ll), hmm.startprob_
for t in range(len(X)):                                  # filtre de Hamilton : P(S_t | infos jusqu'à t)
    w = p * np.exp(ll[t] - ll[t].max())
    a[t] = w / w.sum()
    p = a[t] @ hmm.transmat_
order = np.argsort(sc.inverse_transform(hmm.means_)[:, 1])          # tri par vol : 0 calme, 2 crise
P = pd.DataFrame(a[:, order], index=feat.index, columns=["P0", "P1", "P2"]).shift(1)   # veille de l'événement
e2 = e2.drop(columns=["P0", "P1", "P2"], errors="ignore").join(P.reindex(e2.index, method="ffill"))

d = e2.dropna(subset=["P1"])                              # même échantillon pour les 3 modèles
out = {}
for k, f in {"B": "r5 ~ S * B", "HMM": "r5 ~ S * (P1 + P2)", "B + HMM": "r5 ~ S * (B + P1 + P2)"}.items():
    m = smf.ols(f, data=d).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
    out[k] = pd.concat([m.params, m.pvalues], keys=["coef", "p"])
print(d[["P0", "P1", "P2"]].idxmax(axis=1).value_counts(), "\n")   # nb d'événements par état
pd.DataFrame(out).round(3)

c:\Users\stani\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Le fichier spécifié est introuvable
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\stani\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\stani\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 550, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\stani\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1028, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "

P1    231
P0    172
P2     88
Name: count, dtype: int64 



B    HMM  B + HMM
coef B         -0.003    NaN   -0.008
     Intercept -0.031 -0.034   -0.036
     P1           NaN -0.032   -0.020
     P2           NaN  0.070    0.095
     S         -0.223 -0.173   -0.227
     S:B        0.122    NaN    0.126
     S:P1         NaN  0.005    0.003
     S:P2         NaN -0.019   -0.018
p    B          0.960    NaN    0.885
     Intercept  0.415  0.292    0.390
     P1           NaN  0.528    0.693
     P2           NaN  0.447    0.306
     S          0.000  0.000    0.000
     S:B        0.008    NaN    0.007
     S:P1         NaN  0.919    0.956
     S:P2         NaN  0.778    0.788

In [4]:
# D'où viennent les erreurs hors échantillon ?
train, test = e2.loc[:"2018"], e2.loc["2019":]
m = smf.ols("r5 ~ S", data=train).fit()
t = test.assign(err_mod=(test["r5"] - m.predict(test)) ** 2,
                err_moy=(test["r5"] - train["r5"].mean()) ** 2)
print(t.groupby(t.index.year)[["err_mod", "err_moy"]].sum().round(2), "\n")
t2 = t.drop(t.loc["2020"].index)
print("R2 OOS sans 2020 :", round(1 - t2["err_mod"].sum() / t2["err_moy"].sum(), 4))
print("pente apprise sur 2013-18 :", round(m.params["S"], 3),
      "| vraie pente sur 2019-24 :", round(smf.ols("r5 ~ S", data=test).fit().params["S"], 3))

      err_mod  err_moy
2019    15.66    21.01
2020    31.29    30.19
2021    10.18     8.81
2022    15.55    12.91
2023    12.49     7.00
2024     1.03     1.23 

R2 OOS sans 2020 : -0.0772
pente apprise sur 2013-18 : -0.266 | vraie pente sur 2019-24 : -0.108


In [5]:
# D'où viennent les erreurs hors échantillon ?
train, test = e2.loc[:"2018"], e2.loc["2019":]
m = smf.ols("r5 ~ S", data=train).fit()
t = test.assign(err_mod=(test["r5"] - m.predict(test)) ** 2,
                err_moy=(test["r5"] - train["r5"].mean()) ** 2)
print(t.groupby(t.index.year)[["err_mod", "err_moy"]].sum().round(2), "\n")
t2 = t.drop(t.loc["2020"].index)
print("R2 OOS sans 2020 :", round(1 - t2["err_mod"].sum() / t2["err_moy"].sum(), 4))
print("pente apprise sur 2013-18 :", round(m.params["S"], 3),
      "| vraie pente sur 2019-24 :", round(smf.ols("r5 ~ S", data=test).fit().params["S"], 3))

      err_mod  err_moy
2019    15.66    21.01
2020    31.29    30.19
2021    10.18     8.81
2022    15.55    12.91
2023    12.49     7.00
2024     1.03     1.23 

R2 OOS sans 2020 : -0.0772
pente apprise sur 2013-18 : -0.266 | vraie pente sur 2019-24 : -0.108


In [6]:
# Hors échantillon en fenêtre glissante : chaque année est prédite avec les 3 années d'avant
r2 = lambda o: 1 - ((o.y - o.pred) ** 2).sum() / ((o.y - o.bench) ** 2).sum()
out = {f: [] for f in ["r5 ~ S", "r5 ~ S * B"]}
for y in range(2016, 2025):
    tr, te = e2.loc[str(y - 3):str(y - 1)], e2.loc[str(y)]
    for f in out:
        pred = smf.ols(f, data=tr).fit().predict(te)
        out[f].append(pd.DataFrame({"y": te["r5"], "pred": pred, "bench": tr["r5"].mean()}))
for f, l in out.items():
    o = pd.concat(l)
    print(f"{f:12s} R2 2016-24 = {r2(o):.4f}   R2 2019-24 = {r2(o.loc['2019':]):.4f}")

r5 ~ S       R2 2016-24 = 0.0854   R2 2019-24 = 0.0403
r5 ~ S * B   R2 2016-24 = 0.0764   R2 2019-24 = 0.0189


In [7]:
# Stratégie testable : entrée à 10h35 (après la 1re réaction), sortie à 10h45 / 11h00 / 11h30
COST = 0.04                                         # coût aller-retour en % (spread + glissement) : hypothèse
res = {}
for h in [15, 30, 60]:
    d = e2.assign(y=e2[f"r{h}"] - e2["r5"])         # rendement de 10h35 à la sortie (log-rendements additifs)
    pnl = []
    for yr in range(2016, 2025):                    # même fenêtre glissante de 3 ans que le test précédent
        tr, te = d.loc[str(yr - 3):str(yr - 1)], d.loc[str(yr)]
        pos = np.sign(smf.ols("y ~ S", data=tr).fit().predict(te))    # +1 achat / -1 vente selon la dérive prédite
        pnl.append(pos * te["y"] - COST)
    p = pd.concat(pnl)
    ans = (p.index[-1] - p.index[0]).days / 365.25
    res[f"sortie +{h} min"] = {"trades": len(p), "gain moyen net (%)": p.mean(), "taux de réussite": (p > 0).mean(),
                               "t-stat": p.mean() / p.std() * np.sqrt(len(p)),
                               "Sharpe annuel net": p.mean() / p.std() * np.sqrt(len(p) / ans),
                               "Sharpe annuel brut": (p.mean() + COST) / p.std() * np.sqrt(len(p) / ans)}
pd.DataFrame(res).round(3)

,sortie +15 min,sortie +30 min,sortie +60 min
trades,365.000,365.000,365.000
gain moyen net (%),0.080,0.059,-0.011
taux de réussite,0.526,0.507,0.455
t-stat,2.051,1.123,-0.201
Sharpe annuel net,0.714,0.391,-0.070
Sharpe annuel brut,1.072,0.655,0.195
